<a href="https://colab.research.google.com/github/GauranshPrajapati/HCL-Project-by-Gauransh/blob/main/Gauransh(2026B03110164).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==========================================================
# Project : Bike Rental Demand Analysis
# Industry: Mobility / Smart City
# Goal: Study rental demand by time, season and weather
#       to help a bike-sharing operator plan fleet availability.
# ==========================================================

# ---- Import the libraries we need ----
import pandas as pd      # for working with tables (DataFrames)
import numpy as np       # for number calculations (arrays, percentiles)

# The dataset file (updated to the correct path)
FILE_NAME = "/content/Bike_Rental_Project extended.csv"


# ==========================================================
# TASK 1: Load the dataset and inspect data types
# ==========================================================
print("=" * 60)
print("TASK 1: Load the dataset and inspect data types")
print("=" * 60)

df = pd.read_csv(FILE_NAME)          # read the CSV file into a DataFrame

print("First 5 rows:")
print(df.head())
print("\nNumber of rows and columns:", df.shape)
print("\nData types of each column:")
print(df.dtypes)
# Note: DateTime is stored as text (string), so we will convert it in Task 3.


# ==========================================================
# TASK 2: Check and handle missing humidity readings
# ==========================================================
print("\n" + "=" * 60)
print("TASK 2: Missing humidity readings")
print("=" * 60)

print("Missing values in each column:")
print(df.isnull().sum())

# Fill the missing humidity values with the MEDIAN humidity.
# (Median is safe because it is not affected by very high/low values.)
median_humidity = df["Humidity_Percent"].median()
print("\nMedian humidity used to fill gaps:", median_humidity)

df["Humidity_Percent"] = df["Humidity_Percent"].fillna(median_humidity)

print("Missing humidity values after filling:", df["Humidity_Percent"].isnull().sum())


# ==========================================================
# TASK 3: Convert DateTime to datetime; extract date, month, hour
# ==========================================================
print("\n" + "=" * 60)
print("TASK 3: Convert DateTime and extract date, month, hour")
print("=" * 60)

# Dates look like 01-01-2025 00:00 -> day-month-year, so we tell pandas the format.
df["DateTime"] = pd.to_datetime(df["DateTime"], format="%d-%m-%Y %H:%M")

df["Date"] = df["DateTime"].dt.date      # only the date part
df["Month"] = df["DateTime"].dt.month    # month number (1-12)
df["Hour_Check"] = df["DateTime"].dt.hour  # hour taken from DateTime
# The file already has an "Hour" column, so we compare to make sure they match.
print("Does the Hour column match the DateTime hour?",
      (df["Hour"] == df["Hour_Check"]).all())
df = df.drop(columns=["Hour_Check"])     # we don't need the extra column

print("\nData types after conversion:")
print(df.dtypes)
print("\nSample rows:")
print(df[["DateTime", "Date", "Month", "Hour"]].head())


# ==========================================================
# TASK 4: Calculate total and average rentals
# ==========================================================
print("\n" + "=" * 60)
print("TASK 4: Total and average rentals")
print("=" * 60)

total_rentals = df["Rental_Count"].sum()
average_rentals = df["Rental_Count"].mean()

print("Total rentals   :", total_rentals)
print("Average rentals :", round(average_rentals, 2), "per record")


# ==========================================================
# TASK 5: Find the 10 highest-demand time records
# ==========================================================
print("\n" + "=" * 60)
print("TASK 5: Top 10 highest-demand records")
print("=" * 60)

# nlargest(10, column) gives the 10 rows with the biggest values
top_10 = df.nlargest(10, "Rental_Count")
print(top_10[["DateTime", "Season", "Weather", "Temperature_C", "Rental_Count"]])


# ==========================================================
# TASK 6: Compare average rentals by season
# ==========================================================
print("\n" + "=" * 60)
print("TASK 6: Average rentals by season")
print("=" * 60)

# groupby() splits data into groups, mean() averages each group
season_avg = df.groupby("Season")["Rental_Count"].mean().sort_values(ascending=False)
print(season_avg.round(2))
print("\nBusiest season:", season_avg.idxmax())
print("Quietest season:", season_avg.idxmin())


# ==========================================================
# TASK 7: Compare weekend and weekday demand
# ==========================================================
print("\n" + "=" * 60)
print("TASK 7: Weekend vs weekday demand")
print("=" * 60)

# The Is_Weekend column has "Yes" (weekend) and "No" (weekday)
weekend_avg = df.groupby("Is_Weekend")["Rental_Count"].mean()
print("Average rentals (Yes = weekend, No = weekday):")
print(weekend_avg.round(2))


# ==========================================================
# TASK 8: Compare demand across weather conditions
# ==========================================================
print("\n" + "=" * 60)
print("TASK 8: Average rentals by weather")
print("=" * 60)

weather_avg = df.groupby("Weather")["Rental_Count"].mean().sort_values(ascending=False)
print(weather_avg.round(2))
print("\nBest weather for rentals :", weather_avg.idxmax())
print("Worst weather for rentals:", weather_avg.idxmin())


# ==========================================================
# TASK 9: Analyze rentals by hour of day
# ==========================================================
print("\n" + "=" * 60)
print("TASK 9: Hourly demand summary")
print("=" * 60)

# agg() lets us calculate several statistics at once
hourly_summary = df.groupby("Hour")["Rental_Count"].agg(["mean", "sum", "count"])
print(hourly_summary.round(2))
# Note: this dataset only has two readings per day (hour 0 and hour 12).


# ==========================================================
# TASK 10: Average temperature and rental count using NumPy
# ==========================================================
print("\n" + "=" * 60)
print("TASK 10: NumPy averages")
print("=" * 60)

# Convert the columns into NumPy arrays
temp_array = df["Temperature_C"].to_numpy()
rental_array = df["Rental_Count"].to_numpy()

print("Average temperature (NumPy):", round(np.mean(temp_array), 2), "°C")
print("Average rental count (NumPy):", round(np.mean(rental_array), 2))


# ==========================================================
# TASK 11: Identify unusually high rental counts using IQR
# ==========================================================
print("\n" + "=" * 60)
print("TASK 11: Outliers using the IQR method")
print("=" * 60)

# IQR = Q3 - Q1 (the spread of the middle 50% of the data)
q1 = np.percentile(rental_array, 25)
q3 = np.percentile(rental_array, 75)
iqr = q3 - q1

# Anything above Q3 + 1.5*IQR is "unusually high"
upper_limit = q3 + 1.5 * iqr
lower_limit = q1 - 1.5 * iqr

print("Q1 =", q1, "| Q3 =", q3, "| IQR =", iqr)
print("Upper limit =", upper_limit, "| Lower limit =", lower_limit)

high_outliers = df[df["Rental_Count"] > upper_limit]
print("\nNumber of unusually high records:", len(high_outliers))
print(high_outliers[["DateTime", "Season", "Weather", "Rental_Count"]])


# ==========================================================
# TASK 12: Season-and-weather grouped summary
# ==========================================================
print("\n" + "=" * 60)
print("TASK 12: Season and weather summary")
print("=" * 60)

# Group by TWO columns at once
season_weather = df.groupby(["Season", "Weather"])["Rental_Count"].agg(["mean", "count"])
print(season_weather.round(2))

# A pivot table makes the same result easier to read (seasons as rows)
print("\nPivot table (average rentals):")
print(df.pivot_table(values="Rental_Count", index="Season",
                     columns="Weather", aggfunc="mean").round(2))


# ==========================================================
# TASK 13: Sort records and days by rental demand
# ==========================================================
print("\n" + "=" * 60)
print("TASK 13: Sorting by demand")
print("=" * 60)

# Sort all records from highest to lowest demand
sorted_records = df.sort_values("Rental_Count", ascending=False)
print("Top 5 records:")
print(sorted_records[["DateTime", "Rental_Count"]].head())

# Add up the rentals for each day, then sort the days
daily_total = df.groupby("Date")["Rental_Count"].sum().sort_values(ascending=False)
print("\nTop 5 busiest days:")
print(daily_total.head())
print("\n5 quietest days:")
print(daily_total.tail())


# ==========================================================
# TASK 14: Do temperature and rentals move together? (correlation)
# ==========================================================
print("\n" + "=" * 60)
print("TASK 14: Correlation between temperature and rentals")
print("=" * 60)

# Correlation is between -1 and +1:
#  close to +1 -> move together, close to -1 -> move opposite, close to 0 -> no link
correlation = df["Temperature_C"].corr(df["Rental_Count"])
print("Correlation (temperature vs rentals):", round(correlation, 3))

if abs(correlation) < 0.3:
    print("Meaning: very weak relationship. Temperature barely affects rentals here.")
elif correlation > 0:
    print("Meaning: rentals tend to go UP when temperature goes up.")
else:
    print("Meaning: rentals tend to go DOWN when temperature goes up.")


# ==========================================================
# EXTRA: Data quality check (bonus analysis)
# ==========================================================
print("\n" + "=" * 60)
print("EXTRA: Data quality check")
print("=" * 60)

# dayofweek: Monday = 0 ... Sunday = 6, so 5 and 6 are the real weekend
real_weekend = df["DateTime"].dt.dayofweek >= 5
matches = (real_weekend == (df["Is_Weekend"] == "Yes")).mean()
print("Is_Weekend agrees with the real calendar in", round(matches * 100, 1), "% of rows")

# Does each season appear in several different months? (it shouldn't)
months_per_season = df.groupby("Season")["Month"].nunique()
print("\nNumber of different months each Season label appears in:")
print(months_per_season)
print("Note: if this is ~12 for every season, the Season column does not follow")
print("the calendar, so treat season results with caution.")


# ==========================================================
# SAVE THE CLEANED DATASET (deliverable)
# ==========================================================
df.to_csv("cleaned_bike_rental.csv", index=False)
hourly_summary.to_csv("hourly_demand_summary.csv")
season_avg.to_csv("seasonal_comparison.csv")
weather_avg.to_csv("weather_comparison.csv")
print("\nSaved: cleaned_bike_rental.csv, hourly_demand_summary.csv,")
print("       seasonal_comparison.csv, weather_comparison.csv")

TASK 1: Load the dataset and inspect data types
First 5 rows:
   Record_ID          DateTime  Season     Weather  Temperature_C  \
0          1  01-01-2025 00:00  Winter       Clear           42.0   
1          2  01-01-2025 12:00  Autumn  Heavy Rain           23.1   
2          3  02-01-2025 00:00  Summer      Cloudy           21.0   
3          4  02-01-2025 12:00  Summer      Cloudy           31.1   
4          5  03-01-2025 00:00  Autumn        Rain           25.7   

   Humidity_Percent Is_Weekend  Hour  Rental_Count  
0              83.0        Yes     0            36  
1              65.0        Yes    12            34  
2              85.0         No     0            29  
3              29.0         No    12            33  
4              81.0        Yes     0            34  

Number of rows and columns: (1000, 9)

Data types of each column:
Record_ID             int64
DateTime             object
Season               object
Weather              object
Temperature_C       float6